# Практика: замыкания, декораторы и модуль functools

На этом занятии:

- познакомимся с модулем `functools` и декораторами кэширования `cache` и `lru_cache`;
- реализуем собственный аналог `functools.partial` и рассмотрим стандартный объект `partial`;
- повторим принципы работы с декоратором `wraps`;

In [ ]:
import math
from collections.abc import Callable
from typing import Any

## 1. Практическая задача: фиксируем прибыль

### Условие

Инвестор ежедневно фиксирует результат торгов: прибыль (положительное число) или убыток (отрицательное число). Для оценки динамики требуется средняя прибыль за последние `period` дней.

Реализуйте функцию-фабрику `make_averager`. Она принимает натуральное число `period` и возвращает функцию `get_avg`.

Функция `get_avg` принимает вещественное число `profit` — результат торгов за очередной день — и возвращает вещественное число: среднюю прибыль за последние `period` дней, включая переданный день. Если к моменту вызова зафиксировано `m < period` дней, среднее вычисляется по всем `m` значениям.

Каждая функция, возвращенная вызовом `make_averager`, хранит собственную историю наблюдений независимо от остальных.

### Пример

Для `make_averager(2)`:

| вызов | учитываемые значения | результат |
|---|---|---|
| `get_avg(1)` | `1` | `1.0` |
| `get_avg(2)` | `1, 2` | `1.5` |
| `get_avg(3)` | `2, 3` | `2.5` |
| `get_avg(-3)` | `3, -3` | `0.0` |
| `get_avg(5)` | `-3, 5` | `1.0` |
| `get_avg(5)` | `5, 5` | `5.0` |

### Подсказка

История наблюдений должна сохраняться между вызовами `get_avg`, поэтому ее следует хранить в свободной переменной замыкания. Изменение объекта на месте (например, метод `append` списка) не требует `nonlocal`: он нужен только при перепривязке имени.

In [ ]:
def make_averager(period: int) -> Callable[[float], float]:
    # ваш код
    return lambda profit: 0.0

### Проверка

In [ ]:
test_cases = [
    (2, [1, 2, 3, -3, 5, 5], [1, 1.5, 2.5, 0, 1, 5]),
    (5, [1, 2, 3, 4, 5, -5, -7, -2], [1, 1.5, 2, 2.5, 3, 1.8, 0, -1]),
    (1, [4, -2, 7], [4, -2, 7]),
    (3, [10, 20], [10, 15]),
    (3, [3, 6, 9, 12], [3, 4.5, 6, 9]),
]

for case_number, (period, profits, averages_expected) in enumerate(test_cases):
    get_avg = make_averager(period)

    for profit, average_expected in zip(profits, averages_expected):
        average = get_avg(profit)

        assert math.isclose(average, average_expected, abs_tol=1e-9), (
            f"case {case_number}: {average} != {average_expected}"
        )

first_get_avg = make_averager(2)
second_get_avg = make_averager(2)
first_get_avg(100)
first_get_avg(100)

assert math.isclose(second_get_avg(10), 10), (
    "histories of different averagers must be independent"
)

print("All tests passed")

## 2. Модуль `functools`: кэширование результатов

### Модуль `functools`

Модуль `functools` стандартной библиотеки содержит функции высшего порядка и инструменты для работы с вызываемыми объектами. Значительная часть его содержимого — декораторы и фабрики декораторов, поэтому темы этого занятия (замыкания, декораторы, параметрические декораторы) применяются здесь напрямую. Начнем с декораторов кэширования.

### Мемоизация и декоратор `cache`

**Мемоизация** (*memoization*) — сохранение результата вызова функции для конкретного набора аргументов. При повторном вызове с теми же аргументами функция не выполняется: возвращается сохраненный результат.

Декоратор `functools.cache` хранит результаты в словаре внутри замыкания обертки. Ключом словаря служат аргументы вызова, значением — результат. Поэтому все аргументы декорированной функции должны быть хешируемыми.

In [ ]:
from functools import cache


@cache
def square(number: int) -> int:
    print(f"computing square of {number}")
    return number * number


print(square(4))
# computing square of 4
# 16
print(square(4))  # 16 — тело функции не выполнялось, результат взят из кэша
print(square(5))
# computing square of 5
# 25

Обертка, созданная декоратором `cache`, имеет методы `cache_info()` (статистика обращений к кэшу) и `cache_clear()` (очистка кэша):

- `hits` — число вызовов, для которых результат найден в кэше;
- `misses` — число вызовов, для которых результат пришлось вычислять;
- `maxsize` — максимальный размер кэша (`None` — без ограничения);
- `currsize` — текущее число записей.

In [ ]:
print(square.cache_info())  # CacheInfo(hits=1, misses=2, maxsize=None, currsize=2)
square.cache_clear()
print(square.cache_info())  # CacheInfo(hits=0, misses=0, maxsize=None, currsize=0)

Кэш существенно сокращает время работы, если функция выполняет затратную операцию (длительный расчет, сетевой запрос, обращение к базе данных) и многократно вызывается с одними и теми же аргументами.

Для демонстрации функция `calculate_result` имитирует такую операцию: `time.sleep(2)` приостанавливает выполнение на 2 секунды, после чего функция возвращает результат. Первый вызов с данным аргументом выполняется полностью, повторный возвращает результат из кэша.

In [ ]:
import time
from functools import cache


@cache
def calculate_result(parameter: int) -> int:
    time.sleep(2)  # имитация долгого расчета или сетевого запроса
    return parameter ** 2


start_time = time.perf_counter()
first_result = calculate_result(12)
first_time = time.perf_counter() - start_time
start_time = time.perf_counter()
second_result = calculate_result(12)
second_time = time.perf_counter() - start_time

print(first_result == second_result)  # True
# время зависит от компьютера
print(f"first call: {first_time:.4f} s")
print(f"second call: {second_time:.6f} s")
print(calculate_result.cache_info())
# CacheInfo(hits=1, misses=1, maxsize=None, currsize=1)

### Декоратор `lru_cache`

Декоратор `cache` сохраняет результаты неограниченно долго. Если программа вызывает функцию с большим числом различных аргументов, объем занятой памяти растет без ограничений.

Декоратор `functools.lru_cache` ограничивает размер кэша параметром `maxsize`. Когда кэш заполнен, при добавлении новой записи удаляется запись, к которой обращались позже всех остальных (*least recently used*, LRU). Значение `maxsize` по умолчанию равно `128`; значение `None` снимает ограничение.

`lru_cache` — параметрический декоратор (см. [конспект о параметрических декораторах](conspects/09-parametrized-decorators.md)): запись `@lru_cache(maxsize=2)` сначала вызывает фабрику, а затем применяет возвращенный декоратор к функции. Декоратор `cache` равнозначен `lru_cache(maxsize=None)`.

In [ ]:
from functools import lru_cache


@lru_cache(maxsize=2)
def cube(number: int) -> int:
    print(f"computing cube of {number}")
    return number ** 3


cube(1)                   # computing cube of 1
cube(2)                   # computing cube of 2
cube(1)                   # вывода нет: результат взят из кэша
cube(3)                   # computing cube of 3; результат для 2 вытеснен
cube(2)                   # computing cube of 2; результат для 1 вытеснен
print(cube.cache_info())  # CacheInfo(hits=1, misses=4, maxsize=2, currsize=2)

### Когда кэширование полезно

Кэширование уместно, если выполняются все условия:

- функция **детерминирована и не имеет побочных эффектов**: результат определяется только аргументами;
- функция многократно вызывается с одними и теми же аргументами и имеют место следующие факторы: длительные расчеты, повторные запросы к одним и тем же данным (сеть, база данных), рекурсия с перекрывающимися подзадачами;
- вычисление результата существенно дороже поиска в словаре;
- аргументы хешируемы (числа, строки, хешируемые кортежи);
- при большом числе различных наборов аргументов размер кэша может быть ограничен.

### Когда кэширование неприменимо

- **Побочные эффекты должны выполняться при каждом вызове** (вывод, запись в файл, отправка сетевого запроса, изменение состояния). Тело кэшированной функции выполняется только при первом вызове с данным набором аргументов, поэтому при повторных вызовах побочные эффекты не происходят. В примере выше `print` внутри `square` сработал один раз.
- **Результат зависит от внешнего состояния** (текущее время, `random`, содержимое файла, глобальные переменные): после изменения состояния кэш продолжит возвращать устаревшее значение.
- **Аргументы нехешируемые** (`list`, `dict`, `set`): аргументы вызова служат ключом словаря кэша, поэтому вызов такой функции приводит к `TypeError`.
- **Функция возвращает изменяемый объект**: все вызовы с одинаковыми аргументами получают один и тот же объект, поэтому его изменение искажает содержимое кэша.

In [ ]:
@cache
def get_total(numbers: list[int]) -> int:
    return sum(numbers)


get_total([1, 2, 3])

In [ ]:
@cache
def get_default_items() -> list[int]:
    return [1, 2, 3]


first_items = get_default_items()
first_items.append(4)
print(get_default_items())  # [1, 2, 3, 4] — изменен объект, хранящийся в кэше

## 3. Практическая задача: аналог `functools.partial`

**Частичное применение** (*partial application*) — создание новой функции из существующей путем фиксации значений части ее аргументов. Новая функция принимает только оставшиеся аргументы и вызывает исходную функцию с полным набором.

### Условие

Реализуйте функцию `make_partial`, которая принимает функцию `function`, произвольное число позиционных аргументов `fixed_args` и произвольное число именованных аргументов `fixed_kwargs`. Она возвращает новую функцию. При вызове новой функции с аргументами `args` и `kwargs` вызывается `function`, и ее результат возвращается:

- позиционные аргументы: сначала `fixed_args`, затем `args`;
- именованные аргументы: `fixed_kwargs`, дополненные `kwargs`. Если имя встречается и в `fixed_kwargs`, и в `kwargs`, используется значение из `kwargs` (аргумент, переданный при вызове новой функции).

Исходная функция не изменяется.

### Пример

```python
def power(base: int, exponent: int) -> int:
    return base ** exponent


square = make_partial(power, exponent=2)
two_to_power = make_partial(power, 2)

print(square(5))              # 25, вызывается power(5, exponent=2)
print(two_to_power(10))       # 1024, вызывается power(2, 10)
print(square(5, exponent=3))  # 125, значение при вызове заменяет зафиксированное
```

### Подсказка

Новая функция должна принимать произвольные аргументы (`*args` и `**kwargs`) и использовать `function`, `fixed_args` и `fixed_kwargs` как свободные переменные замыкания. Словари объединяются записью `{**fixed_kwargs, **kwargs}`: при совпадении ключей приоритет у последнего словаря.

In [ ]:
def make_partial(
    function: Callable[..., Any],
    *fixed_args: Any,
    **fixed_kwargs: Any,
) -> Callable[..., Any]:
    # ваш код
    return function

### Проверка

In [ ]:
def power(base: int, exponent: int) -> int:
    return base ** exponent


def join_words(*words: str, separator: str = " ") -> str:
    return separator.join(words)


square = make_partial(power, exponent=2)
two_to_power = make_partial(power, 2)
hello_joiner = make_partial(join_words, "Hello")
comma_joiner = make_partial(join_words, separator=", ")

assert square(5) == 25
assert square(base=3) == 9
assert square(5, exponent=3) == 125
assert two_to_power(10) == 1024
assert two_to_power(exponent=3) == 8
assert hello_joiner("world") == "Hello world"
assert hello_joiner("big", "world", separator=", ") == "Hello, big, world"
assert comma_joiner("first", "second", "third") == "first, second, third"
assert comma_joiner("first", "second", separator="") == "firstsecond"
assert make_partial(power)(2, 3) == 8
assert make_partial(int, base=2)("101") == 5
assert make_partial(max, key=len)(["ab", "abc", "a"]) == "abc"

assert square(5) == 25, "repeated calls must not accumulate arguments"
assert hello_joiner("again") == "Hello again", (
    "repeated calls must not accumulate arguments"
)
assert power(2, 3) == 8, "the original function must not be changed"

print("All tests passed")

## 4. Объект `functools.partial`

### `partial`

В модуле `functools` имеется готовая реализация частичного применения — `partial(function, /, *args, **keywords)`. Ее семантика совпадает с реализованной в задаче: зафиксированные позиционные аргументы ставятся перед переданными при вызове, а именованные объединяются, причем значения, переданные при вызове, приоритетнее.

In [ ]:
from functools import partial


def power(base: int, exponent: int) -> int:
    return base ** exponent


square = partial(power, exponent=2)
two_to_power = partial(power, 2)

print(square(5))                    # 25
print(two_to_power(10))             # 1024
print(square(5, exponent=3))        # 125
print(partial(int, base=2)("101"))  # 5

Вызов `partial` возвращает не функцию, а объект класса `functools.partial` — вызываемый объект, который хранит исходную функцию и зафиксированные аргументы в атрибутах `func`, `args` и `keywords`. В отличие от функции, у него нет атрибута `__name__`.

In [ ]:
print(type(square))                 # <class 'functools.partial'>
print(square.func is power)         # True
print(square.args)                  # ()
print(square.keywords)              # {'exponent': 2}
print(two_to_power.args)            # (2,)
print(hasattr(square, "__name__"))  # False

### `partial` и `lambda`

Частичное применение можно записать и через `lambda`, но семантика различается. Объект `partial` сохраняет **значения** аргументов в момент создания. Lambda-функция обращается к внешним именам **в момент вызова** (см. [практику 7 занятия](../lesson07/practice.ipynb)), поэтому после изменения переменной она вернет другой результат. (Присваивание lambda-функции переменной здесь выполнено только для сравнения.)

In [ ]:
exponent = 2
square_lambda = lambda base: power(base, exponent)
square_partial = partial(power, exponent=exponent)
exponent = 3

print(square_lambda(5))   # 125 — значение exponent прочитано в момент вызова
print(square_partial(5))  # 25 — значение 2 зафиксировано при создании

Типичные применения `partial` — настройка функции для передачи в качестве аргумента (параметр `key`, обработчики событий) и создание специализированных вариантов универсальной функции. Например, функция расстояния до заданного числа из [практики 7](../lesson07/practice.ipynb) с `partial` вместо `lambda`:

In [ ]:
def get_distance(number: int, target: int) -> int:
    return abs(number - target)


numbers = [3, 8, 14, 11, 20]
print_csv = partial(print, sep=", ")

print(min(numbers, key=partial(get_distance, target=10)))  # 11
print_csv(1, 2, 3)                                         # 1, 2, 3

## 5. Повторение: `wraps`

Декоратор возвращает обертку, поэтому без дополнительных мер декорированная функция теряет собственные метаданные: ее `__name__` и `__doc__` становятся именем и документацией обертки. Это затрудняет отладку, а инструменты, использующие эти атрибуты, получают неверные сведения.

Декоратор `functools.wraps(function)` копирует в обертку атрибуты исходной функции (`__module__`, `__name__`, `__qualname__`, `__doc__`, аннотации).

In [ ]:
from functools import wraps


def log_call(function: Callable[..., Any]) -> Callable[..., Any]:
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        print(f"calling {function.__name__}")
        return function(*args, **kwargs)

    return wrapper


def log_call_with_wraps(function: Callable[..., Any]) -> Callable[..., Any]:
    @wraps(function)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        print(f"calling {function.__name__}")
        return function(*args, **kwargs)

    return wrapper


def add(first_term: int, second_term: int) -> int:
    """Return the sum of two numbers."""
    return first_term + second_term


plain_add = log_call(add)
wrapped_add = log_call_with_wraps(add)

print(f"{plain_add.__name__}: {plain_add.__doc__}")
# wrapper: None
print(f"{wrapped_add.__name__}: {wrapped_add.__doc__}")
# add: Return the sum of two numbers.

## 6. Другие инструменты `functools`

В модуле `functools` есть и другие полезные объекты. Полный перечень приведен в [документации](https://docs.python.org/3/library/functools.html). Некоторые из них:

- `reduce` — свертка последовательности функцией двух аргументов в единственное значение;
- `cmp_to_key` — преобразование функции сравнения двух элементов в функцию для параметра `key`;
- `singledispatch` — выбор реализации функции по типу первого аргумента;
- `total_ordering`, `cached_property`, `partialmethod` — инструменты для работы с классами, к которым вернемся после их изучения.

## 7. Практическая задача: регистрация функций в словаре

В [практике 7 занятия](../lesson07/practice.ipynb) функции-стратегии скидок передавались в `get_discounted_price` списком, который формировался вручную:

```python
all_discounts = [
    get_loyalty_discount,
    get_item_amount_discount,
    get_general_amount_discount,
]
```

У такого подхода есть недостаток: определение стратегии и ее добавление в список разнесены в коде. При добавлении новой стратегии легко забыть обновить список. Вместо этого функцию можно регистрировать в словаре декоратором **в момент определения** (при выполнении `def` декоратор применяется сразу, см. [конспект о моменте декорирования](conspects/06-decoration-time.md)).

### Условие

Реализуйте параметрический декоратор `register_function(registry, key="")` со следующими параметрами:

- `registry` — словарь, в который регистрируется функция; обязательный параметр;
- `key` — строка, под которой функция регистрируется; необязательный параметр, по умолчанию пустая строка. Если ключ не передан (равен пустой строке), в качестве ключа используется имя декорируемой функции.

Декоратор вычисляет ключ и проверяет, есть ли в словаре запись с этим ключом:

- если запись есть, декоратор не изменяет словарь;
- если записи нет, декоратор добавляет в словарь запись: ключ — вычисленный ключ, значение — декорируемая функция.

В обоих случаях декоратор возвращает исходную функцию без изменений (не обертку).

### Пример

```python
handlers = {}


@register_function(handlers)
def say_hello() -> str:
    return "hello"


@register_function(handlers, "bye")
def say_goodbye() -> str:
    return "goodbye"


@register_function(handlers, "bye")
def say_bye() -> str:
    return "bye"


print(list(handlers))                  # ['say_hello', 'bye']
print(handlers["bye"] is say_goodbye)  # True — запись с ключом "bye" уже существовала
print(say_hello())                     # hello — функция не изменилась
```

In [ ]:
def register_function(
    registry: dict[str, Callable[..., Any]],
    key: str = "",
) -> Callable[[Callable[..., Any]], Callable[..., Any]]:
    # ваш код
    return lambda function: function

### Проверка

In [ ]:
registry: dict[str, Callable[..., Any]] = {}


@register_function(registry)
def first_function() -> int:
    return 1


@register_function(registry, "second")
def another_name() -> int:
    return 2


@register_function(registry, key="third")
def third_function() -> int:
    return 3


assert list(registry) == ["first_function", "second", "third"]
assert registry["first_function"] is first_function
assert registry["second"] is another_name
assert registry["third"] is third_function
assert first_function() == 1
assert another_name() == 2
assert another_name.__name__ == "another_name"


@register_function(registry, "second")
def duplicate_with_key() -> int:
    return 20


@register_function(registry)
def first_function() -> int:
    return 10


assert len(registry) == 3
assert registry["second"]() == 2
assert registry["first_function"]() == 1
assert duplicate_with_key() == 20
assert first_function() == 10

other_registry: dict[str, Callable[..., Any]] = {}


@register_function(other_registry)
def standalone() -> int:
    return 0


assert list(other_registry) == ["standalone"]
assert "standalone" not in registry
assert len(registry) == 3

print("All tests passed")